In [1]:
!nvidia-smi

Wed May  6 23:46:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%%writefile vector_add.cu

#include <iostream>
using namespace std;

__global__ void add(int *a, int *b, int *c)
{
    int i = threadIdx.x;
    c[i] = a[i] + b[i];
}

int main()
{
    int a[5] = {1,2,3,4,5};
    int b[5] = {10,20,30,40,50};
    int c[5];

    int *d_a, *d_b, *d_c;

    cudaMalloc((void**)&d_a, 5*sizeof(int));
    cudaMalloc((void**)&d_b, 5*sizeof(int));
    cudaMalloc((void**)&d_c, 5*sizeof(int));

    cudaMemcpy(d_a, a, 5*sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, 5*sizeof(int), cudaMemcpyHostToDevice);

    add<<<1,5>>>(d_a, d_b, d_c);

    cudaMemcpy(c, d_c, 5*sizeof(int), cudaMemcpyDeviceToHost);

    cout<<"Result:"<<endl;

    for(int i=0;i<5;i++)
        cout<<c[i]<<" ";

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}

Writing vector_add.cu


In [3]:
!nvcc vector_add.cu -o vector_add

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [4]:
!./vector_add

Result:
11 22 33 44 55 